# Carrinho IESB — Marcos 1 e 2 juntos

Adaptação simples do gabarito da Aula 10: mapa, A*, distância, tempo e agente Qwen no mesmo notebook. Execute de cima para baixo. O professor permanece em arquivo separado.

**Estimativas didáticas:** distâncias aproximadas da planta; velocidade constante escolhida pelo usuário, sem paradas, aceleração ou espera. 10 e 15 km/h são cenários de cálculo, não velocidades medidas do carrinho. O tempo calculado por distância/velocidade não é um modelo treinado.

O grafo mantém 70 nós e 69 ligações confirmadas na versão anterior, incluindo Mesa 1 a Mesa 4. Ele é uma árvore: há apenas um caminho simples entre dois locais. O A* minimiza os metros desse grafo simplificado; não demonstra escolha entre rotas físicas alternativas. Os itens acadêmicos pendentes estão em `O_QUE_FALTA.md`.

## 1. Origem, destino e velocidade
Altere as três variáveis abaixo e execute novamente a partir daqui. Para usar 15 km/h, troque `10.0` por `15.0`.

In [ ]:
ORIGEM = "Entrada Principal"
DESTINO = "Mesa 4"
VELOCIDADE_KMH = 10.0

## 2. Mapa e distâncias aproximadas em metros
Referência: planta enviada em 12/11/2026. As medidas são estimativas, não levantamento topográfico.

- Eixo principal: 210 m divididos em 60 + 48 + 39 + 39 + 24 m, usando aproximadamente a posição vertical dos acessos no desenho.
- Pátio: referência 55 m até o eixo e 53 m de comprimento; ponto central a 27,5 m do eixo. Mesas colocadas por hipótese a 5, 10, 15 e 20 m desse ponto.
- Design, C e B: referência 40 × 20 m; nó do corredor próximo da entrada direita. Distâncias internas estimadas pela posição das salas ao longo dos 40 m, incluindo entrada na sala.
- I/J: referência longitudinal de aproximadamente 75 m; nó do corredor na altura do acesso lateral. A largura está “a conferir”: os 12 m do acesso e os deslocamentos laterais internos são hipóteses.
- Cinema/Biblioteca e apoios: estimativas de circulação compatíveis com a disposição do desenho, sem medir portas individualmente. A biblioteca recebe 60 m, incluindo deslocamento ao norte e passagem até sua entrada.

O croqui tem proporções inconsistentes entre áreas; não se aplica uma única escala de pixels a todo o mapa. Os nós que têm apenas o nome do bloco são pontos de referência 1 m após o corredor. Em cada tupla abaixo, o terceiro número é o comprimento estimado do trecho. As conexões continuam bidirecionais. O rótulo D-02 do Bloco C continua cadastrado provisoriamente como C-02.

In [ ]:
LOCAIS = [
    "Entrada Principal", "Eixo - Norte", "Eixo - Design", "Eixo - Bloco C",
    "Eixo - Bloco B", "Eixo - Fundo",
    "Pátio/Direito", "Mesa 1", "Mesa 2", "Mesa 3", "Mesa 4",
    "Sala dos Professores", "Gastro", "Oasis Smart",
    "WC Acessível Pátio", "WC Feminino Pátio",
    "Cinema", "WC Cinema", "Anexo Creche", "Biblioteca",
    "Design", "Corredor Design", "D-01", "D-02", "D-03", "D-04", "D-05", "D-06",
    "WC Masculino Design", "WC Feminino Design",
    "Bloco C", "Corredor Bloco C", "C-01", "C-02", "C-03", "C-04", "C-05", "C-06",
    "WC Masculino C", "WC Feminino C",
    "Bloco B", "Corredor Bloco B", "B-01", "B-02", "B-03", "B-04", "B-05", "B-06",
    "WC Masculino B", "WC Feminino B",
    "Acesso Lateral I/J", "Corredor I/J", "Bloco I", "Bloco J",
    "I-01", "I-02", "I-03", "I-04", "I-05", "I-06",
    "J-01", "J-02", "J-03", "J-04", "J-05", "J-06",
    "WC Masculino I", "WC Feminino I", "WC Masculino J", "WC Feminino J",
]


# origem, destino, distância estimada em metros
LIGACOES = [
    ("Entrada Principal", "Eixo - Norte", 60),
    ("Eixo - Norte", "Eixo - Design", 48),
    ("Eixo - Design", "Eixo - Bloco C", 39),
    ("Eixo - Bloco C", "Eixo - Bloco B", 39),
    ("Eixo - Bloco B", "Eixo - Fundo", 24),
    ("Eixo - Norte", "Pátio/Direito", 27.5),
    ("Eixo - Norte", "Cinema", 15),
    ("Eixo - Norte", "Biblioteca", 60),
    ("Eixo - Design", "Corredor Design", 5),
    ("Eixo - Bloco C", "Corredor Bloco C", 5),
    ("Eixo - Bloco B", "Corredor Bloco B", 5),
    ("Eixo - Bloco C", "Acesso Lateral I/J", 5),
    ("Acesso Lateral I/J", "Corredor I/J", 12),
    ("Corredor Design", "Design", 1),
    ("Corredor Bloco C", "Bloco C", 1),
    ("Corredor Bloco B", "Bloco B", 1),
    ("Corredor I/J", "Bloco I", 1),
    ("Corredor I/J", "Bloco J", 1),
]

for local, metros in [
    ("Mesa 1", 5), ("Mesa 2", 10), ("Mesa 3", 15), ("Mesa 4", 20),
    ("Sala dos Professores", 15), ("Gastro", 20), ("Oasis Smart", 30),
    ("WC Acessível Pátio", 40), ("WC Feminino Pátio", 35),
]:
    LIGACOES.append(("Pátio/Direito", local, metros))

for local, metros in [("WC Cinema", 20), ("Anexo Creche", 25)]:
    LIGACOES.append(("Cinema", local, metros))

for corredor, prefixo, banheiro in [
    ("Corredor Design", "D", "Design"),
    ("Corredor Bloco C", "C", "C"),
    ("Corredor Bloco B", "B", "B"),
]:
    for sala, metros in [("01", 35), ("02", 25), ("03", 15),
                         ("04", 35), ("05", 25), ("06", 8)]:
        LIGACOES.append((corredor, prefixo + "-" + sala, metros))
    LIGACOES.append((corredor, "WC Masculino " + banheiro, 4))
    LIGACOES.append((corredor, "WC Feminino " + banheiro, 7))

for local, metros in [
    ("I-01", 37), ("I-02", 28), ("I-03", 16),
    ("I-04", 37), ("I-05", 29), ("I-06", 20),
    ("J-01", 13), ("J-02", 25), ("J-03", 36),
    ("J-04", 13), ("J-05", 22), ("J-06", 30),
    ("WC Masculino I", 13), ("WC Feminino I", 8),
    ("WC Masculino J", 36), ("WC Feminino J", 40),
]:
    LIGACOES.append(("Corredor I/J", local, metros))

VIZINHOS = {}
for local in LOCAIS:
    VIZINHOS[local] = []
for origem, destino, metros in LIGACOES:
    VIZINHOS[origem].append((destino, metros))
    VIZINHOS[destino].append((origem, metros))

print(len(LOCAIS), "nós e", len(LIGACOES), "ligações com distância em metros.")

## 3. A* e cálculo do tempo
A estrutura do A* segue o gabarito. O custo acumulado agora soma metros. A heurística `h = 0` é admissível e faz a busca se comportar como custo uniforme. Uma heurística informada e a comparação de buscas ainda estão pendentes no Marco 1.

Conversão: `m/s = km/h ÷ 3,6`; `tempo (s) = distância (m) × 3,6 ÷ velocidade (km/h)`. A velocidade é a mesma em todas as ligações, portanto minimizar metros também minimiza o tempo. Alterar somente a velocidade não muda o caminho.

In [ ]:
def h(local, destino):
    return 0


def a_estrela(origem, destino):
    if origem not in VIZINHOS or destino not in VIZINHOS:
        return None, None
    fronteira = [(h(origem, destino), 0, origem, [origem])]
    visitados = []
    while len(fronteira) > 0:
        melhor = min(fronteira)
        fronteira.remove(melhor)
        f, g, local, caminho = melhor
        if local == destino:
            return caminho, g
        if local in visitados:
            continue
        visitados.append(local)
        for vizinho, custo in VIZINHOS[local]:
            if vizinho not in visitados:
                novo_g = g + custo
                fronteira.append((novo_g + h(vizinho, destino), novo_g,
                                  vizinho, caminho + [vizinho]))
    return None, None



def calcular_tempo(distancia_m, velocidade_kmh):
    if not isinstance(velocidade_kmh, (int, float)) or isinstance(velocidade_kmh, bool):
        raise ValueError("A velocidade precisa ser um número maior que zero.")
    if not 0 < velocidade_kmh < float("inf"):
        raise ValueError("A velocidade precisa ser finita e maior que zero.")
    if not 0 <= distancia_m < float("inf"):
        raise ValueError("A distância precisa ser finita e não negativa.")
    return distancia_m * 3.6 / velocidade_kmh

## 4. Ferramenta de rota usada pelo Qwen
A mesma função serve para a consulta direta e para a chamada do LLM. Ela devolve o caminho, a distância aproximada e o tempo estimado em segundos e minutos. O LLM escolhe a ferramenta; o Python calcula a rota e o tempo.

In [ ]:
def calcular_rota(origem: str, destino: str, velocidade_kmh: float = VELOCIDADE_KMH) -> str:
    """
    Calcula com A* a menor distância estimada no grafo e o tempo a velocidade constante.

    Args:
        origem: nome exato do local de partida na lista de locais
        destino: nome exato do local de chegada na lista de locais
        velocidade_kmh: velocidade constante em km/h, positiva; omitir para usar a configuração
    """
    if origem not in LOCAIS or destino not in LOCAIS:
        return "Erro: origem ou destino não existe no mapa. Consulte a lista LOCAIS."
    try:
        calcular_tempo(0, velocidade_kmh)
    except ValueError as erro:
        return "Erro: " + str(erro)
    caminho, distancia = a_estrela(origem, destino)
    if caminho is None:
        return "Não existe caminho entre " + origem + " e " + destino + "."
    segundos = calcular_tempo(distancia, velocidade_kmh)
    return ("Caminho: " + " -> ".join(caminho)
            + f". Distância aproximada: {distancia:.1f} m."
            + f" Velocidade constante: {velocidade_kmh:g} km/h."
            + f" Tempo estimado: {segundos:.1f} s ({segundos / 60:.2f} min), sem paradas.")


def ferramenta_do_trio(origem: str, destino: str, velocidade_kmh: float = VELOCIDADE_KMH) -> str:
    """
    Consulta a rota A* e retorna distância aproximada e tempo estimado de deslocamento.

    Args:
        origem: nome exato do local de partida na lista de locais
        destino: nome exato do local de chegada na lista de locais
        velocidade_kmh: velocidade constante em km/h, positiva; omitir para usar a configuração
    """
    return calcular_rota(origem, destino, velocidade_kmh)


def listar_vizinhos(local: str) -> str:
    """
    Lista os locais ligados diretamente a um local da planta do IESB.

    Args:
        local: local consultado, com nome igual ao da lista de locais
    """
    if local not in LOCAIS:
        return "Erro: o local " + str(local) + " não existe no mapa."
    partes = []
    for vizinho, custo in VIZINHOS[local]:
        partes.append(vizinho)
    return "Vizinhos de " + local + ": " + ", ".join(partes) + "."



FERRAMENTAS = [ferramenta_do_trio, listar_vizinhos]

## 5. Resultado e comparação entre 10 e 15 km/h
Esta parte funciona sem carregar o Qwen. A primeira consulta usa a velocidade configurada no início. A tabela compara as duas velocidades na mesma rota.

In [ ]:
print(calcular_rota(ORIGEM, DESTINO, VELOCIDADE_KMH))
caminho, distancia = a_estrela(ORIGEM, DESTINO)
if caminho is not None:
    print("\nVelocidade | Distância estimada | Tempo estimado")
    for velocidade in [10, 15]:
        segundos = calcular_tempo(distancia, velocidade)
        print(f"{velocidade:>8} km/h | {distancia:>10.1f} m | {segundos:.1f} s ({segundos / 60:.2f} min)")

for origem, destino in [("Entrada Principal", "Mesa 1"),
                        ("Biblioteca", "C-02"), ("B-04", "I-05")]:
    print(calcular_rota(origem, destino, VELOCIDADE_KMH))

## 6. Qwen do gabarito
A partir daqui é necessário instalar `transformers` e baixar o modelo público. No Colab, use uma GPU T4 se disponível. As funções anteriores já calculam os resultados sem o LLM.

In [ ]:
%pip -q install transformers

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    dispositivo = "cuda"
else:
    dispositivo = "cpu"
print("Dispositivo:", dispositivo)

nome_qwen = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizador_qwen = AutoTokenizer.from_pretrained(nome_qwen)
if dispositivo == "cuda":
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float16).to(dispositivo)
else:
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float32)
qwen.eval()


def conversar(mensagens, ferramentas=None, max_tokens=200):
    entrada = tokenizador_qwen.apply_chat_template(
        mensagens, tools=ferramentas, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(dispositivo)
    with torch.no_grad():
        saida = qwen.generate(**entrada, max_new_tokens=max_tokens, do_sample=False)
    tamanho_da_pergunta = entrada["input_ids"].shape[-1]
    return tokenizador_qwen.decode(saida[0][tamanho_da_pergunta:], skip_special_tokens=True)


print("Qwen carregado.")

## 7. Agente com ferramentas — mesmo ciclo do gabarito

In [ ]:
REGRAS = (
    "Você ajuda a encontrar rotas no térreo do IESB. "
    "Para perguntas sobre caminhos, custos e locais vizinhos, use as ferramentas. "
    "Nunca invente rotas, distâncias em metros ou acessos: use apenas o resultado das ferramentas. "
    "As distâncias são aproximadas em metros e os tempos são estimados a velocidade constante, sem paradas. Use velocidade_kmh quando a pergunta informar uma velocidade. "
    "Responda em português, de forma curta. "
    "Nomes válidos dos locais: " + ", ".join(LOCAIS) + "."
)


def extrair_json(texto):
    inicio = texto.find("{")
    fim = texto.rfind("}")
    if inicio == -1 or fim == -1:
        return None
    try:
        return json.loads(texto[inicio:fim + 1])
    except (ValueError, TypeError):
        return None


def ler_pedido_de_ferramenta(resposta):
    if "<tool_call>" not in resposta:
        return None
    return extrair_json(resposta)


def executar(pedido):
    if not isinstance(pedido, dict):
        return "Erro: pedido de ferramenta inválido."
    nome = pedido.get("name")
    argumentos = pedido.get("arguments", {})
    if not isinstance(argumentos, dict):
        return "Erro: argumentos da ferramenta inválidos."
    if nome == "calcular_rota":
        return calcular_rota(argumentos.get("origem"), argumentos.get("destino"),
                             argumentos.get("velocidade_kmh", VELOCIDADE_KMH))
    if nome == "ferramenta_do_trio":
        return ferramenta_do_trio(argumentos.get("origem"), argumentos.get("destino"),
                             argumentos.get("velocidade_kmh", VELOCIDADE_KMH))
    if nome == "listar_vizinhos":
        return listar_vizinhos(argumentos.get("local"))
    return "Erro: a ferramenta " + str(nome) + " não existe."


def agente(pergunta, mostrar=True):
    mensagens = [
        {"role": "system", "content": REGRAS},
        {"role": "user", "content": pergunta},
    ]
    for passo in range(3):
        resposta = conversar(mensagens, FERRAMENTAS)
        pedido = ler_pedido_de_ferramenta(resposta)
        if pedido is None:
            return resposta
        resultado = executar(pedido)
        if mostrar:
            print("   [ferramenta]", pedido.get("name"), pedido.get("arguments"))
            print("   [resultado ]", resultado)
        mensagens.append({"role": "assistant", "content": resposta})
        mensagens.append({"role": "tool", "content": resultado})
    return "Não consegui concluir em 3 passos."

## 8. Perguntas e descrição da ferramenta

In [ ]:
from transformers.utils import get_json_schema

print(json.dumps(get_json_schema(ferramenta_do_trio), indent=1, ensure_ascii=False))

perguntas = [
    "Como vou da Entrada Principal até a Mesa 4 e quanto tempo levo a 10 km/h?",
    "Qual a distância de B-04 até I-05 e o tempo a 15 km/h?",
    "Quais locais estão ligados ao Pátio/Direito?",
]
for pergunta in perguntas:
    print("PERGUNTA:", pergunta)
    print("RESPOSTA:", agente(pergunta))
    print()

## 9. Avaliação inicial da escolha da ferramenta
Os seis casos herdados verificam somente a primeira ferramenta escolhida. Não substituem a avaliação de pelo menos 15 pedidos com argumentos e resultados esperados exigida na entrega final.

In [ ]:
CASOS = [
    ("Como vou da Entrada Principal até a Mesa 2?", "ferramenta_do_trio"),
    ("Qual é a rota de B-04 para I-05?", "ferramenta_do_trio"),
    ("Quais locais estão ligados ao Pátio/Direito?", "listar_vizinhos"),
    ("O que há ao lado do Corredor Design?", "listar_vizinhos"),
    ("Bom dia!", None),
    ("Obrigado pela ajuda!", None),
]

acertos = 0
for pergunta, esperada in CASOS:
    mensagens = [{"role": "system", "content": REGRAS},
                 {"role": "user", "content": pergunta}]
    pedido = ler_pedido_de_ferramenta(conversar(mensagens, FERRAMENTAS))
    if pedido is None:
        escolhida = None
    else:
        escolhida = pedido.get("name")
    if escolhida == esperada:
        acertos = acertos + 1
        marca = "certo "
    else:
        marca = "ERRADO"
    print(marca, "|", pergunta, "| esperada:", esperada, "| escolhida:", escolhida)

print("Escolheu a ferramenta certa em", acertos, "de", len(CASOS), "casos.")

## Limites e próximos passos
Este notebook reúne o código atual dos Marcos 1 e 2. Ainda faltam a comparação de buscas em três tamanhos de mapa, a heurística própria informada, o modelo de custo aprendido com comparação de dois modelos, a avaliação completa do agente e os relatórios. A fórmula distância/velocidade adicionada aqui é uma estimativa determinística, não aprendizado de máquina. Consulte `O_QUE_FALTA.md`.

O grafo agrega corredores em um único ponto; viagens entre salas podem voltar por esse ponto. Isso limita a precisão física. As ligações e as distâncias precisam ser revisadas com medições para representar o campus com maior fidelidade.